# Sesión 1 · Ejercicio 2 — Mi primer generador
**Objetivo:** entrenar un autocodificador variacional (VAE), muestrearlo
e interpolar en su espacio latente.
**Tiempo:** MÍNIMO 25 min · COMPLETO 60 min
**Produce:** §2 de su bitácora — rejilla de muestras + interpolación
**Necesitas:** §1 listo (el setup ya activo)

> Hoy TODOS entrenan sobre el mismo conjunto de demostración de
> imágenes. Es deliberado: mañana entrenarán una GAN sobre ese mismo
> conjunto y la comparación será directa. (Sobre sus propios datos:
> sección EXTENSIÓN.)


### Cómo trabajar este cuaderno (1 minuto de lectura)

1. **Guarde su copia**: Archivo → Guardar una copia en Drive. Si no, pierde su trabajo al cerrar.
2. Ejecute las celdas **en orden**. Solo las marcadas `#### OBLIGATORIO ####` producen su entregable; las de **EXTENSIÓN** son opcionales, para quien le sobre tiempo.
3. ¿Algo no corre, o tarda demasiado? Ejecute la **CELDA DE RESCATE**: carga resultados ya calculados y usted sigue con el análisis. Usarla **no descuenta puntos** — solo dígalo en su bitácora.
4. Al terminar, copie la figura y sus observaciones (2-3 líneas con sus palabras) a la sección de su **bitácora** que dice el encabezado. Eso es TODO el entregable — no se pide nada más.


In [ ]:
#### OBLIGATORIO #### — setup (idempotente: puede ejecutarla dos veces)
import os, sys
if not os.path.isdir("src"):
    if not os.path.isdir("IAA6_M13_Gen"):
        !git clone -q https://github.com/AdriannaGmz/IAA6_M13_Gen
    %cd IAA6_M13_Gen
!pip install -q -r requirements.txt
sys.path.insert(0, ".")
from src import datos, modelos, evaluar, graficas, rescate
MODO_GPU = rescate.hay_gpu()   # imprime "GPU disponible" o "Modo CPU"


In [ ]:
#### OBLIGATORIO #### — el conjunto común
X, y, meta = datos.cargar("imagen")     # conjunto de demostración
datos.ficha(meta)


In [ ]:
#### OBLIGATORIO #### — instanciar el modelo
vae = modelos.VAE(meta, dim_latente=32)
print("VAE listo. Espacio latente de", vae.dim_latente, "dimensiones.")


In [ ]:
#### OBLIGATORIO #### — los dos términos de la pérdida, a mano
# Antes de entrenar, calcule sobre un lote los DOS términos que el
# entrenamiento va a optimizar. Son los de la secuencia de diapositivas
# del Bloque 2.
import torch
import torch.nn.functional as F

x = X[:128].to(vae.dispositivo)
with torch.no_grad():
    mu, logvar = vae.codificador(x)
    # truco de reparametrización: z = mu + sigma * epsilon
    z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
    x_rec = vae.decodificador(z)

    # COMPLETAR: término de reconstrucción
    # Pista: compare x_rec con x usando F.mse_loss(..., reduction="sum")
    #        y divida entre len(x)
    perdida_rec = ...

    # COMPLETAR: término de regularización
    # Pista: -0.5 * torch.sum(1 + logvar - mu**2 - exp(logvar)),
    #        dividido entre len(x)
    perdida_reg = ...

print(f"Reconstrucción: {float(perdida_rec):8.1f}")
print(f"Regularización: {float(perdida_reg):8.1f}")
print("La pérdida del VAE es la SUMA de ambos, en tensión permanente.")


In [ ]:
#### OBLIGATORIO #### — entrenar (≈ 10-30 s; en CPU tarda un poco más)
historial = vae.entrenar(
    X, epocas=10,
    cb=lambda e, r: print(f"  época {e + 1:2d}/10 · pérdida {r['total']:8.1f}"))
graficas.curva(historial, "Mi primer generador")


In [ ]:
# ── CELDA DE RESCATE ────────────────────────────────────────
# ¿No entrenó? ¿Se desconectó? Ejecute esto y siga.
contenido, figuras = rescate.cargar("s1_vae")
vae = modelos.VAE(meta, dim_latente=32)
vae.codificador.load_state_dict(contenido["state_dicts"]["codificador"])
vae.decodificador.load_state_dict(contenido["state_dicts"]["decodificador"])
historial = contenido["historial"]


In [ ]:
#### OBLIGATORIO #### — muestrear
muestras = vae.muestrear(16)
fig = graficas.rejilla(muestras, "16 muestras de mi VAE")
fig.savefig("bitacora_s1e2_rejilla.png", dpi=120)


In [ ]:
#### RECOMENDADO #### — interpolar entre dos latentes
# COMPLETAR: codifique dos imágenes reales y recorra la línea recta
# entre sus latentes.
# Pista: vae.codificar(X[:2]) devuelve dos latentes; páselos a
#        graficas.interpolacion(vae, ..., ..., pasos=10)
z = ...
fig = ...
fig.savefig("bitacora_s1e2_interpolacion.png", dpi=120)


### Observación — complete ANTES de cerrar (2 renglones) · OBLIGATORIO

- ¿La pérdida bajó de manera sostenida? ___
- ¿Cómo se ven las muestras: nítidas o borrosas? ___

Anote ambas respuestas tal cual en su bitácora (§2).
**No busque la explicación todavía: llega mañana.**


In [ ]:
#### OBLIGATORIO #### — artefacto para la bitácora
print("Copie este bloque en la sección §2 de su bitácora y adjunte")
print("bitacora_s1e2_rejilla.png y bitacora_s1e2_interpolacion.png:\n")
print(f"- VAE de {vae.dim_latente} dimensiones latentes, 10 épocas")
print(f"- Pérdida final: {historial['total'][-1]:.1f} "
      f"(inicial: {historial['total'][0]:.1f})")
print("- La pérdida bajó: <sí/no>")
print("- Las muestras se ven: <nítidas/borrosas>")


### EXTENSIÓN (equipos rápidos)
1. **Aritmética latente:** codifique todas las muestras de dos clases,
   calcule el latente promedio de cada una, y decodifique puntos sobre
   la línea que los une. ¿En qué punto una figura se convierte en otra?
2. **Sus datos:** entrene el VAE sobre su propio conjunto
   (`datos.cargar(su_modo, su_fuente)`). Todo lo demás es idéntico:
   ésa es la gracia de la capa común.


In [ ]:
#### EXTENSIÓN #### — aritmética latente (esqueleto)
idx_a = (y == 0).nonzero(as_tuple=True)[0][:100]
idx_b = (y == 1).nonzero(as_tuple=True)[0][:100]
z_a = vae.codificar(X[idx_a]).mean(dim=0)
z_b = vae.codificar(X[idx_b]).mean(dim=0)
graficas.interpolacion(vae, z_a, z_b, pasos=10)
